# 02: COLMAP SfM

Google Drive に画像を配置してから実行してください。
先頭セルの `SCENE_NAME` を変更するだけで OK です。

In [ ]:
# ========== ユーザー設定（ここだけ変更） ==========
SCENE_NAME = "my_scene"
# ================================================

from google.colab import drive
from pathlib import Path
import pycolmap

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/gaussian_splatting")
IMAGE_DIR  = DRIVE_ROOT / "input" / SCENE_NAME / "images"
OUTPUT_DIR = DRIVE_ROOT / "output" / SCENE_NAME
SPARSE_DIR = OUTPUT_DIR / "sparse" / "0"
IMAGES_OUT = OUTPUT_DIR / "images"
DB_PATH    = OUTPUT_DIR / "colmap.db"

assert IMAGE_DIR.exists(), (
    f"画像ディレクトリが見つかりません: {IMAGE_DIR}\n"
    f"Drive に {IMAGE_DIR} を作成して画像を置いてください。"
)
images = (
    list(IMAGE_DIR.glob("*.jpg"))
    + list(IMAGE_DIR.glob("*.jpeg"))
    + list(IMAGE_DIR.glob("*.png"))
)
assert len(images) >= 10, f"画像が {len(images)} 枚しかありません（10 枚以上必要）"
print(f"入力画像: {len(images)} 枚")

In [ ]:
for d in [OUTPUT_DIR, SPARSE_DIR, IMAGES_OUT]:
    d.mkdir(parents=True, exist_ok=True)
print("ディレクトリ準備完了")

In [ ]:
pycolmap.extract_features(
    database_path=str(DB_PATH),
    image_path=str(IMAGE_DIR),
    camera_model="OPENCV",
    sift_options={"max_num_features": 8192},
)
print("特徴点抽出: 完了")

In [ ]:
num_images = len(images)
if num_images <= 50:
    pycolmap.match_exhaustive(database_path=str(DB_PATH))
    print("Exhaustive マッチング: 完了")
else:
    pycolmap.match_sequential(database_path=str(DB_PATH), overlap=10)
    print("Sequential マッチング: 完了")

In [ ]:
import shutil

maps = pycolmap.incremental_mapping(
    database_path=str(DB_PATH),
    image_path=str(IMAGE_DIR),
    output_path=str(OUTPUT_DIR / "sparse"),
)

if not maps:
    raise RuntimeError(
        "再構成に失敗しました。\n"
        "・画像の重複を増やす（隣接フレームで 30〜60% の重複率）\n"
        "・画像数を増やす（30 枚以上推奨）\n"
        "docs/troubleshooting.md も参照してください。"
    )

best_key = max(maps, key=lambda k: maps[k].num_reg_images())
reconstruction = maps[best_key]
print(f"再構成成功: {reconstruction.num_reg_images()} カメラ / {reconstruction.num_points3D()} 点")

In [ ]:
reconstruction.write_binary(str(SPARSE_DIR))
print(f"sparse 結果を保存: {SPARSE_DIR}")
for fname in ["cameras.bin", "images.bin", "points3D.bin"]:
    p = SPARSE_DIR / fname
    print(f"  {fname}: {p.stat().st_size:,} bytes")

In [ ]:
copied = 0
for img in IMAGE_DIR.iterdir():
    if img.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        dst = IMAGES_OUT / img.name
        if not dst.exists():
            shutil.copy2(str(img), str(dst))
            copied += 1
print(f"{copied} 枚の画像を {IMAGES_OUT} にコピーしました")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

pts  = np.array([p.xyz for p in reconstruction.points3D.values()])
cams = np.array([img.tvec for img in reconstruction.images.values()])

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
step = max(1, len(pts) // 5000)
ax.scatter(pts[::step, 0], pts[::step, 1], pts[::step, 2], s=0.5, c="gray", alpha=0.3)
ax.scatter(cams[:, 0], cams[:, 1], cams[:, 2], s=30, c="red", marker="^", label="cameras")
ax.set_title(f"SfM: {len(pts):,} pts / {len(cams)} cameras")
ax.legend()
plt.tight_layout()

preview_path = OUTPUT_DIR / "sfm_preview.png"
plt.savefig(str(preview_path), dpi=100)
plt.show()
print(f"プレビュー保存: {preview_path}")
print("\n03_train_3dgs.ipynb に進んでください。")